# BinSense — M2: EDA + Splits

**Goal of this notebook:** close out Milestone 2. Turn the 3,874 downloaded
metadata JSONs into tidy tables, run the Week 2&3 EDA (item / weight / volume /
items-per-bin / image quality), and — most importantly — **write the
`seed` / `extend` / `eval` split manifests** that the whole self-training
architecture depends on.

Outputs written to `data/splits/`:
- `bins_master.csv`   — one row per bin (the tidy source of truth)
- `items_master.csv`  — one row per (bin, ASIN) line item
- `image_quality.csv` — per-image blur/brightness/contrast/resolution + poor-quality flag
- `seed.csv`          — single-ASIN bins (clean labels) → embedder seed + FAISS gallery
- `extend.csv`        — multi-ASIN bins → pseudo-labeling targets
- `eval.csv`          — held-out gold candidates (detection + identity), excluded from seed/extend

Runtime: **Colab-first** (Drive-mounted). Also runs locally via `env_utils` path resolution.


In [ ]:
# Cell 1: Bootstrap — path resolution + Colab setup
#
# Code (utils/, tools/, notebooks/) lives in the git repo.
# Data (images/, metadata/, splits/) lives on Google Drive.
# BINSENSE_DATA_DIR tells env_utils where the data root is when they differ.

import sys, os
from pathlib import Path

# ── Colab branch ──────────────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')

    _DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'

    # Code root: once the GitHub remote is set, replace with a git clone:
    #   !git clone https://github.com/<user>/binsense /content/binsense
    #   PROJECT_ROOT = Path('/content/binsense')
    PROJECT_ROOT = Path(_DRIVE_ROOT)

    # Data root — always on Drive (images + metadata are not in git)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(_DRIVE_ROOT) / 'data')

    import subprocess
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + ['seaborn', 'opencv-python-headless', 'tqdm'],
        check=True,
    )

# ── Local branch (D:/GitHub_Repo/Amazon BinSense) ────────────────────────
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    # Set BINSENSE_DATA_DIR if data lives elsewhere, e.g.:
    #   os.environ['BINSENSE_DATA_DIR'] = r'G:\My Drive\...\data'
    # Leave unset to fall back to PROJECT_ROOT/data.

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Running in  : {"Google Colab" if IN_COLAB else "Local"}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA DIR    : {os.getenv("BINSENSE_DATA_DIR", str(PROJECT_ROOT / "data"))}')
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"



In [ ]:
# ── Cell 2: Imports + path config ──────────────────────────────────────────
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from utils.env_utils import setup_env

cfg = setup_env(verbose=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# Reproducibility for the split sampling later
RANDOM_SEED = 42

print(f'\nMetadata dir : {cfg.metadata_dir}')
print(f'Images dir   : {cfg.images_dir}')
print(f'Splits dir   : {cfg.splits_dir}')
assert cfg.metadata_dir.exists(), f"Metadata dir missing: {cfg.metadata_dir}"

## Step 1 — Build tidy tables from metadata

Each JSON has `BIN_FCSKU_DATA` (ASIN → {name, quantity, weight, length/width/height})
and a bin-level `EXPECTED_QUANTITY` (total item count). We flatten this into:

- **`df_items`** — one row per (bin, ASIN): the line-item grain.
- **`df_bins`**  — one row per bin: aggregates used for EDA and splitting.


In [ ]:
# ── Cell 3: Parse all metadata JSONs into df_items + df_bins ────────────────
def _dim(d, key):
    """Return numeric value of a {'unit','value'} dimension block, or NaN."""
    v = (d.get(key) or {})
    return v.get("value", np.nan) if isinstance(v, dict) else np.nan

meta_files = sorted(cfg.metadata_dir.glob("*.json"))
print(f"Parsing {len(meta_files)} metadata files ...")

item_rows, bin_rows = [], []
for p in tqdm(meta_files):
    bin_id = p.stem
    try:
        meta = json.loads(p.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"  skip {bin_id}: {e}")
        continue

    fcsku = meta.get("BIN_FCSKU_DATA") or {}
    expected_qty = meta.get("EXPECTED_QUANTITY")

    asins = list(fcsku.keys())
    for asin, d in fcsku.items():
        item_rows.append({
            "bin_id": bin_id,
            "asin": asin,
            "name": d.get("name"),
            "quantity": d.get("quantity"),
            "weight": _dim(d, "weight"),
            "length": _dim(d, "length"),
            "width":  _dim(d, "width"),
            "height": _dim(d, "height"),
        })

    qty_sum = sum((d.get("quantity") or 0) for d in fcsku.values())
    bin_rows.append({
        "bin_id": bin_id,
        "asin_count": len(asins),
        "expected_quantity": expected_qty,
        "quantity_sum": qty_sum,
        "asin_list": "|".join(asins),
    })

df_items = pd.DataFrame(item_rows)
df_bins  = pd.DataFrame(bin_rows)

# Derived item geometry
df_items["volume"] = df_items["length"] * df_items["width"] * df_items["height"]

print(f"\ndf_items: {df_items.shape}  |  df_bins: {df_bins.shape}")
print(f"Unique ASINs (catalog size): {df_items['asin'].nunique()}")
df_bins.head()

## Step 2 — Items per bin: single vs. multi (the architecture driver)

This is the distribution that forced the metric-learning + self-training design:
single-ASIN bins are the *seed*; multi-ASIN bins must be pseudo-labeled for coverage.


In [ ]:
# ── Cell 4: Distinct product types per bin + expected quantity ──────────────
counts = df_bins["asin_count"].value_counts().sort_index()
pct = (counts / counts.sum() * 100).round(1)
dist = pd.DataFrame({"bins": counts, "pct": pct})
print("Distinct ASINs per bin:\n", dist, "\n")

n_single = int((df_bins["asin_count"] == 1).sum())
n_multi  = int((df_bins["asin_count"] >= 2).sum())
print(f"Single-ASIN bins: {n_single}  ({n_single/len(df_bins)*100:.1f}%)")
print(f"Multi-ASIN  bins: {n_multi}  ({n_multi/len(df_bins)*100:.1f}%)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(x=dist.index, y=dist["bins"], ax=ax[0], color="steelblue")
ax[0].set(title="Distinct product types per bin", xlabel="# distinct ASINs", ylabel="# bins")
sns.histplot(df_bins["expected_quantity"].dropna(), bins=range(0, 22), ax=ax[1], color="indianred")
ax[1].set(title="Expected total quantity per bin", xlabel="EXPECTED_QUANTITY", ylabel="# bins")
plt.tight_layout(); plt.show()

## Step 3 — Catalog / item-frequency analysis

How often each ASIN appears across bins. Near-one-shot tail confirms retrieval,
not classification.


In [ ]:
# ── Cell 5: ASIN frequency + coverage ──────────────────────────────────────
asin_freq = df_items.groupby("asin")["bin_id"].nunique().sort_values(ascending=False)
print(f"Unique ASINs            : {asin_freq.size}")
print(f"ASINs appearing in 1 bin: {(asin_freq == 1).sum()}  "
      f"({(asin_freq == 1).mean()*100:.1f}%)")
print(f"Median bins per ASIN    : {asin_freq.median():.0f}")

# Seed coverage: ASINs reachable from single-ASIN bins only
single_bins = df_bins[df_bins["asin_count"] == 1]
seed_asins = set("|".join(single_bins["asin_list"]).split("|")) - {""}
print(f"\nSeed ASINs (from single-ASIN bins): {len(seed_asins)} "
      f"({len(seed_asins)/asin_freq.size*100:.1f}% of catalog)")

top = (df_items.groupby(["asin", "name"])["bin_id"].nunique()
       .sort_values(ascending=False).head(10))
print("\nTop 10 items by # bins:"); print(top)

plt.figure(figsize=(10, 4))
sns.histplot(asin_freq.values, bins=range(1, asin_freq.max() + 2), color="seagreen")
plt.yscale("log"); plt.xlabel("# bins an ASIN appears in"); plt.ylabel("# ASINs (log)")
plt.title("ASIN frequency (long-tail → near one-shot)"); plt.show()

## Step 4 — Weight & volume analysis

Item weight/volume distributions, biggest items, and whether weight correlates
with volume (sanity check on the metadata).


In [ ]:
# ── Cell 6: Weight, volume, correlation ────────────────────────────────────
clean = df_items.dropna(subset=["weight", "volume"])
clean = clean[(clean["weight"] > 0) & (clean["volume"] > 0)]

print("Weight (lbs) describe:\n", clean["weight"].describe(), "\n")
print("Volume (cu in) describe:\n", clean["volume"].describe())

top_vol = (clean.sort_values("volume", ascending=False)
           .drop_duplicates("asin").head(5)[["asin", "name", "volume", "weight"]])
print("\nTop 5 biggest items by volume:"); print(top_vol)

corr = clean["weight"].corr(np.log1p(clean["volume"]))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(np.log1p(clean["weight"]), bins=40, ax=ax[0], color="darkorange")
ax[0].set(title="log(1+weight)", xlabel="log lbs")
sns.histplot(np.log1p(clean["volume"]), bins=40, ax=ax[1], color="purple")
ax[1].set(title="log(1+volume)", xlabel="log cu-in")
ax[2].scatter(np.log1p(clean["volume"]), clean["weight"], s=5, alpha=0.2)
ax[2].set(title=f"weight vs log-volume (r={corr:.2f})", xlabel="log volume", ylabel="weight")
plt.tight_layout(); plt.show()

## Step 5 — Image quality pass

Compute per-image metrics (resolution, brightness, contrast, blur via Laplacian
variance) and flag likely-poor images. This informs labeling priority and is
saved for later filtering.

> Reads every JPG from Drive — a few minutes on Colab. Set `SAMPLE_N` to a small
> int for a quick dry run, or `None` to process all images.


In [ ]:
# ── Cell 7: Per-image quality metrics ──────────────────────────────────────
import cv2

SAMPLE_N = None  # set e.g. 200 for a quick dry run; None = all images

img_files = sorted(cfg.images_dir.glob("*.jpg"))
if SAMPLE_N:
    img_files = img_files[:SAMPLE_N]
print(f"Scanning {len(img_files)} images ...")

rows = []
for p in tqdm(img_files):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None:
        rows.append({"bin_id": p.stem, "readable": False})
        continue
    h, w = img.shape
    rows.append({
        "bin_id": p.stem, "readable": True,
        "width": w, "height": h,
        "brightness": float(img.mean()),
        "contrast": float(img.std()),
        "blur_var": float(cv2.Laplacian(img, cv2.CV_64F).var()),
    })

df_img = pd.DataFrame(rows)

# Flag poor quality: very blurry OR too dark/bright OR low contrast
ok = df_img["readable"] == True
blur_thr = df_img.loc[ok, "blur_var"].quantile(0.05)
df_img["poor_quality"] = (
    (~ok) |
    (df_img["blur_var"] < blur_thr) |
    (df_img["brightness"] < 30) | (df_img["brightness"] > 225) |
    (df_img["contrast"] < 20)
)
print(f"\nFlagged poor-quality: {int(df_img['poor_quality'].sum())} "
      f"/ {len(df_img)}  (blur_var < {blur_thr:.1f})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df_img.loc[ok, "blur_var"], bins=60, ax=ax[0], color="teal")
ax[0].axvline(blur_thr, color="red", ls="--"); ax[0].set(title="Blur (Laplacian var)", xlim=(0, df_img.loc[ok,"blur_var"].quantile(0.99)))
sns.histplot(df_img.loc[ok, "brightness"], bins=60, ax=ax[1], color="goldenrod")
ax[1].set(title="Brightness"); plt.tight_layout(); plt.show()

## Step 6 — Build the seed / extend / eval splits

**Rules (this is the part everything downstream depends on):**

1. **`eval`** is carved out *first* and held out from all training. Stratified across
   ASIN-count buckets so it covers single + multi + dense bins. Within eval,
   multi-ASIN bins (`asin_count >= 2`) are the **identity** gold candidates; all eval
   bins are **detection** gold candidates.
2. **`seed`** = single-ASIN bins **not** in eval (clean labels → embedder + FAISS gallery).
3. **`extend`** = multi-ASIN bins **not** in eval (pseudo-labeling targets).

Sizes are deliberate: ~100+ detection gold, ~40+ identity gold (per PROJECT_PLAN §6).
Everything is seeded (`RANDOM_SEED`) for reproducibility.


In [ ]:
# ── Cell 8: Carve eval holdout, then seed / extend ─────────────────────────
rng = np.random.default_rng(RANDOM_SEED)

def bucket(n):
    if n == 1: return "single"
    if n in (2, 3): return "multi"
    if n >= 4: return "hard"
    return "unknown"

df_bins["bucket"] = df_bins["asin_count"].map(bucket)

# How many of each bucket to hold out for eval (gold candidates)
EVAL_TARGET = {"single": 30, "multi": 70, "hard": 30}

eval_ids = []
for b, k in EVAL_TARGET.items():
    pool = df_bins.loc[df_bins["bucket"] == b, "bin_id"].to_numpy()
    take = min(k, len(pool))
    eval_ids.extend(rng.choice(pool, size=take, replace=False).tolist())
eval_ids = set(eval_ids)

# asin_count == 0 means empty/corrupt metadata — drop from all splits.
df_bins["split"] = np.select(
    [
        df_bins["asin_count"] == 0,
        df_bins["bin_id"].isin(eval_ids),
        df_bins["asin_count"] == 1,
    ],
    ["drop", "eval", "seed"],
    default="extend",
)
n_drop = int((df_bins["split"] == "drop").sum())
if n_drop:
    print(f"Dropped {n_drop} bin(s) with empty metadata (asin_count == 0)\n")

# eval sub-roles
is_eval = df_bins["split"] == "eval"
df_bins["gold_detection"] = is_eval
df_bins["gold_identity"]  = is_eval & (df_bins["asin_count"] >= 2)

summary = df_bins["split"].value_counts()
print("Split sizes:\n", summary, "\n")
print(f"  eval detection gold : {int(df_bins['gold_detection'].sum())}")
print(f"  eval identity gold  : {int(df_bins['gold_identity'].sum())}")
print(f"  seed ASINs (unique) : "
      f"{len(set('|'.join(df_bins.loc[df_bins.split=='seed','asin_list']).split('|')) - {''})}")

## Step 7 — Persist all artifacts

In [ ]:
# ── Cell 9: Write artifacts to data/splits/ ────────────────────────────────
cfg.splits_dir.mkdir(parents=True, exist_ok=True)

cols = ["bin_id", "asin_count", "expected_quantity", "quantity_sum",
        "bucket", "split", "gold_detection", "gold_identity", "asin_list"]

df_bins[cols].to_csv(cfg.splits_dir / "bins_master.csv", index=False)
df_items.to_csv(cfg.splits_dir / "items_master.csv", index=False)
df_img.to_csv(cfg.splits_dir / "image_quality.csv", index=False)

df_bins.loc[df_bins.split == "seed",   cols].to_csv(cfg.splits_dir / "seed.csv",   index=False)
df_bins.loc[df_bins.split == "extend", cols].to_csv(cfg.splits_dir / "extend.csv", index=False)
df_bins.loc[df_bins.split == "eval",   cols].to_csv(cfg.splits_dir / "eval.csv",   index=False)

for f in ["bins_master.csv", "items_master.csv", "image_quality.csv",
          "seed.csv", "extend.csv", "eval.csv"]:
    pth = cfg.splits_dir / f
    print(f"  wrote {pth.name:20s} ({pth.stat().st_size/1024:.1f} KB)")

print("\nM2 complete. Splits are the source of truth for labeling (M3) and "
      "embedder/gallery (M5). eval.csv is held out — never train on it.")